# GINからVG1GC-66のNWBを取得し、downsampled+ophysのみの軽量版にしてDriveへ保存

`docs/data.md` の実測どおり、GINのNWB（1ファイル約1.7GB）の内訳は `acquisition`（5kHz生波形16ch、約2.3GB論理サイズ、82%）、
`processing`（499MB、18%）。さらに`processing`は`behavior`（動画ネイティブレート、約302MB）・`downsampled`（30Hz、約130MB）・
`ophys`（30Hz、約65MB）に分かれる。

`bdbc_nwb_explorer.read_nwb()` はデフォルト(`downsampled=True`)では `processing['downsampled']` と `processing['ophys']` しか読まず、
`acquisition` と `processing['behavior']`（ネイティブレートのキーポイント）は一切参照しない
（`bdbc_nwb_explorer/view.py` の `read_acquisition()` / `read_video_tracking()` / `read_trials()` / `read_roi_dFF()` で確認済み）。
そのためこの2つを除去した軽量版（**約195MB**）でも既存パイプライン（`src/glmhmm_ver4.py`）は無改造で動く。

**このノートブックはWSLローカル実行専用**（Colabは使わない。理由は計画ファイル参照: GINのgit-annex転送はColabの一時ディスク・帯域と相性が悪く、無料枠のセッション時間制限もある）。

## 決定事項
- 対象: `VG1GC-66` の `task-day1`〜`task-day15`（GIN上に存在しない`task-day2`は自動でスキップされる）
- `acquisition` と `processing/behavior` を除去し、`processing/downsampled` と `processing/ophys` だけを残す
  （将来ネイティブレート約100Hzのキーポイントが必要になった場合はGINから再取得すればよい、という前提での破棄）
- 変換後、GIN由来のフル版NWBは**破棄**する（Driveにフル版は残さない）
- 既存の `nwb_manual/VG1GC-66/..._task-day15.nwb`（フル版）も今回軽量版化して統一する
- 再実行時、Driveに変換済みファイルが既にあるdayはスキップする（冪等）
- **1day処理するごとに、その日の生NWBを即座に削除**してから次のdayに進む（15日分を先にまとめて取得すると25GB超が一時的に溜まるため）

## day1実行で判明した実務上の注意点

day1を実際に処理した際に以下が起きたため、このノートブックと`src/nwb_shrink.py`に対処を組み込んである:

- **HTTPS cloneが接続タイムアウトすることがある**（port 443が繋がらない）。実行した時間帯にGINサーバー側が落ちていたためと見られ、恒常的なブロックではない。`shrink.clone_gin_dataset()`がHTTPSを試した上でSSHにフォールバックする。SSH経由には事前にGINアカウント作成＋SSH公開鍵の登録が必要（次のセル参照）。
- **SSH URLは`ssh://git@gin.g-node.org/ORG/REPO.git`形式が必須**。scp風の`git@gin.g-node.org:ORG/REPO.git`形式は`GIN: Invalid repository path`で失敗する。
- **`datalad get`がダウンロード完了後にハングすることがある**（原因はCドライブの空き容量枯渇と見られる。実測で50分近くかかったうち、実転送は約31分でその後約17分ハング）。`shrink.datalad_get_with_recovery()`がタイムアウト後にファイルサイズ照合で救済する。加えて、各day処理前にディスク空き容量を事前チェックしてハングそのものを予防する。
- **Google Drive（9pマウント）への書き込みは`shutil.copy2`等の`sendfile()`経由だと`OSError: [Errno 5] Input/output error`になることがある**（Cドライブ枯渇時に発生）。`shrink.copy_to_drive()`が通常のread/writeでコピーし、失敗時は中途半端なファイルを残さない。
- **変換後の一時ファイル名は`{mouse_id}_{date}_{task_day}.nwb`形式である必要がある**（`bdbc_nwb_explorer`がファイル名からセッションIDを解析するため）。`shrink.normalized_dest_filename()`で決めた名前を最初から一時ファイルに使う。
- **WSLのext4.vhdxはdropしても自動で縮まない**。WSL内で消してもホストCドライブの空きは戻らない。
  取り込み完了後にWindows側で `wsl --shutdown` してから
  `Optimize-VHD -Path <ext4.vhdx> -Mode Full`（Hyper-Vが無い環境では `diskpart` の
  `select vdisk file=...` → `compact vdisk`）を実行して縮める。
- **`check_free_space`はDriveのクラウドクォータを見ていない**。`shutil.disk_usage`が9pマウント越しに
  返すのはローカル/仮想値なので、Drive側の残量はGoogleドライブのUIで別途確認する
  （保存量は全task-day分でも約2.5GBなので通常は問題にならない）。


In [1]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

assert "WSL_DISTRO_NAME" in os.environ, (
    "このノートブックはWSLローカル実行を想定しています。"
    "Colab/Windowsネイティブでの実行は未対応です（datalad/git-annexが前提のため）。"
)

import config
import src.glmhmm_ver4 as v4
import src.nwb_shrink as shrink

print("DATA_NWB_ROOT:", config.DATA_NWB_ROOT)
print("DATA_NWB_ROOTが存在するか:", config.DATA_NWB_ROOT.exists())


環境: ローカル (WSL)
DATA_CSV_ROOT: /mnt/g/.shortcut-targets-by-id/1fI6PWRHgihU6asA4OyW-_rN-JII33Fkj/hackathon_data
DATA_CSV_BACKUP_ROOT: /mnt/g/マイドライブ/braidyn-bc-backup/hackathon_data


DATA_NWB_ROOT: /mnt/g/マイドライブ/nwb_manual
DATA_NWB_ROOTが存在するか: True


## 依存関係の確認

`datalad` と `git-annex` が必要。**このノートブックからは自動インストールしない**（sudoが要るapt操作を伴うため）。
無ければ事前にWSLのターミナルで以下を実行しておく:

```bash
sudo apt-get update && sudo apt-get install -y git-annex
/mnt/c/Users/<user>/braidyn-bc/.venv-wsl/bin/pip install datalad
```

### SSHフォールバック用の準備（HTTPS cloneが失敗した場合に必要）

HTTPSでのcloneが失敗した場合（GINサーバー側の一時的なダウン等）、`shrink.clone_gin_dataset()`は自動でSSHにフォールバックする。SSH経由には以下が必要:

1. SSH鍵を生成（未作成なら）:
   ```bash
   ssh-keygen -t ed25519 -f ~/.ssh/gin_ed25519 -N '' -C 'braidyn-bc-gin'
   cat ~/.ssh/gin_ed25519.pub
   ```
2. https://gin.g-node.org/user/sign_up でアカウント作成
3. ログイン後 Settings → SSH Keys → Add Key で上記公開鍵を登録
4. `~/.ssh/config` に以下を追記:
   ```
   Host gin.g-node.org
     HostName gin.g-node.org
     User git
     IdentityFile ~/.ssh/gin_ed25519
     IdentitiesOnly yes
   ```
5. 疎通確認: `ssh -T git@gin.g-node.org`（"successfully authenticated" と出ればOK）


In [2]:
for tool in ("datalad", "git-annex"):
    path = shutil.which(tool)
    status = path if path else "見つかりません"
    print(f"{tool}: {status}")


datalad: /mnt/c/Users/dmasu/braidyn-bc/.venv-wsl/bin/datalad
git-annex: /usr/bin/git-annex


In [3]:
MOUSE_ID = "VG1GC-66"
TARGET_DAYS = [f"task-day{n}" for n in range(1, 16)]  # day1〜day15（GINに無いdayは自動スキップ）

GIN_HTTPS_URL = "https://gin.g-node.org/BraiDyn-BC/Kondo2025_CuedLeverPullNWB"
# scp風の "git@gin.g-node.org:ORG/REPO.git" は "GIN: Invalid repository path" になる。
# ssh://形式が必須（実測で確認済み）。
GIN_SSH_URL = "ssh://git@gin.g-node.org/BraiDyn-BC/Kondo2025_CuedLeverPullNWB.git"

# リポジトリ外の作業ディレクトリ(.gitignore不要)。datalad clone(メタデータのみ)を置く。
GIN_CACHE_DIR = Path.home() / "gin_cache" / "Kondo2025_CuedLeverPullNWB"
WORK_TMP_DIR = Path.home() / "gin_cache" / "_tmp_processing_only"

DEST_ROOT = config.DATA_NWB_ROOT / MOUSE_ID

# 1dayあたり生NWB約1.7GB + 変換後約200MBの余裕を見た閾値。
# Cドライブ枯渇でdatalad get/dropがハングしたりDrive書き込みがI/Oエラーになる
# 現象を実測したため、処理前に必ずチェックする。
MIN_FREE_BYTES_WSL_HOME = 4 * 1024 ** 3   # GIN_CACHE_DIR側（生NWBの受け皿）
MIN_FREE_BYTES_DRIVE = 1 * 1024 ** 3      # DEST_ROOT側（変換後ファイルの保存先）

print("GIN_CACHE_DIR:", GIN_CACHE_DIR)
print("WORK_TMP_DIR:", WORK_TMP_DIR)
print("DEST_ROOT:", DEST_ROOT)


GIN_CACHE_DIR: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB
WORK_TMP_DIR: /home/takejin/gin_cache/_tmp_processing_only
DEST_ROOT: /mnt/g/マイドライブ/nwb_manual/VG1GC-66


## GINデータセットのclone（メタデータのみ）

`datalad clone` はファイル名・ディレクトリ構造だけを取得し、NWB本体（git-annexの実体）は取得しない。
既にcloneしてあれば再cloneしない。

まずHTTPSを試し、接続できない場合（サーバー側の一時的なダウン等）は自動でSSHにフォールバックする
（`shrink.clone_gin_dataset()`。SSHには事前のアカウント作成・鍵登録が必要、上の依存関係セル参照）。


In [4]:
if not GIN_CACHE_DIR.exists():
    shrink.clone_gin_dataset(GIN_CACHE_DIR, GIN_HTTPS_URL, GIN_SSH_URL)
else:
    print("既にcloneされています:", GIN_CACHE_DIR)


既にcloneされています: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB


## 構造確認（GIN内の実際のパス・命名規則）

GIN側のディレクトリ構造（`VG1GC-66_..._task-dayN.nwb` のようにローカルの `nwb_manual` と同じ命名か、
BIDS風 `sub-XX/ses-XX` かなどは未確認。`src/nwb_shrink.find_gin_nwb_path()` は
mouse_id・task_dayを含むファイル名を再帰的に探すため、事前に厳密な構造を知らなくても動く想定だが、
**このセルを実行して実際に見つかるか必ず確認してから**次のセル（本体取得ループ）に進むこと。
見つからない場合はここで `gin_root.rglob("*.nwb")` の出力を見て、`find_gin_nwb_path` の
マッチ条件を実際の命名規則に合わせて調整する。


In [5]:
print("GIN内の全NWBファイル数:", len(list(GIN_CACHE_DIR.rglob("*.nwb"))))
print("---")
for day in TARGET_DAYS:
    found = shrink.find_gin_nwb_path(GIN_CACHE_DIR, MOUSE_ID, day)
    print(f"{day}: {found}")


GIN内の全NWBファイル数: 495
---
task-day1: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/data/VG1GC-66/VG1GC-66_2023-08-21_task-day1.nwb
task-day2: None
task-day3: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/data/VG1GC-66/VG1GC-66_2023-08-23_task-day3.nwb
task-day4: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/data/VG1GC-66/VG1GC-66_2023-08-24_task-day4.nwb
task-day5: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/data/VG1GC-66/VG1GC-66_2023-08-25_task-day5.nwb
task-day6: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/data/VG1GC-66/VG1GC-66_2023-08-28_task-day6.nwb
task-day7: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/data/VG1GC-66/VG1GC-66_2023-08-29_task-day7.nwb
task-day8: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/data/VG1GC-66/VG1GC-66_2023-08-30_task-day8.nwb


task-day9: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/data/VG1GC-66/VG1GC-66_2023-08-31_task-day9.nwb
task-day10: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/data/VG1GC-66/VG1GC-66_2023-09-01_task-day10.nwb
task-day11: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/data/VG1GC-66/VG1GC-66_2023-09-04_task-day11.nwb
task-day12: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/data/VG1GC-66/VG1GC-66_2023-09-05_task-day12.nwb
task-day13: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/data/VG1GC-66/VG1GC-66_2023-09-06_task-day13.nwb
task-day14: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/data/VG1GC-66/VG1GC-66_2023-09-07_task-day14.nwb
task-day15: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/data/VG1GC-66/VG1GC-66_2023-09-08_task-day15.nwb


## day1〜day15を1日ずつ取得→変換→即削除

Driveに既にあるdayはスキップする。1day処理し終えるごとに、その日の生NWB（GINから取得した実体）を
`datalad drop` で即座に削除してから次のdayへ進む（ローカルのピーク使用量を1day分に抑えるため）。

各dayの処理前にディスク空き容量をチェックする（`check_free_space`）。取得は
`datalad_get_with_recovery`（タイムアウト+サイズ照合で救済）、Driveへのコピーは
`copy_to_drive`（sendfile経由のI/Oエラーを回避）、dropは`datalad_drop_with_recovery`
（タイムアウト時は直接削除）を使う。一時ファイル名は最初から`normalized_dest_filename`で
決めた正規の名前にする（bdbc_nwb_explorerのセッションID解析に必要）。

1dayあたりの目安時間: 実転送が約30分（サーバー帯域依存、短縮不可）+ 変換約1.5分 +
検証・コピー数分。ディスク空き容量が十分であれば、以前発生したような
（転送完了後に十数分以上ハングする）追加の待ち時間は起きないはず。

ループの前に、前回の中断runが残した一時ファイルを `cleanup_work_dir` で掃除する（1ファイル約195MBがWSLのext4.vhdxを膨らませ続けるため）。


In [6]:
# 前回の中断runが残した一時ファイルを掃除してからループに入る
shrink.cleanup_work_dir(WORK_TMP_DIR)

削除対象の一時ファイルはありません: /home/takejin/gin_cache/_tmp_processing_only


0

In [7]:
WORK_TMP_DIR.mkdir(parents=True, exist_ok=True)
DEST_ROOT.mkdir(parents=True, exist_ok=True)

summary = []

for day in TARGET_DAYS:
    print(f"\n=== {day} ===")

    existing = v4.find_nwb_file(MOUSE_ID, day)
    if existing is not None:
        print(f"既にDriveに存在するためスキップ: {existing}")
        summary.append((day, "skipped (already exists)", existing))
        continue

    src_path = shrink.find_gin_nwb_path(GIN_CACHE_DIR, MOUSE_ID, day)
    if src_path is None:
        print(f"GIN内で{MOUSE_ID} {day}のNWBが見つかりませんでした。スキップします。")
        summary.append((day, "skipped (not found on GIN)", None))
        continue

    shrink.check_free_space(GIN_CACHE_DIR, MIN_FREE_BYTES_WSL_HOME, label="WSL側・生NWB受け皿")
    shrink.check_free_space(DEST_ROOT, MIN_FREE_BYTES_DRIVE, label="Drive側・変換後保存先")

    print(f"GIN上のパス: {src_path}")
    print("datalad get 実行中...")
    rel_path = src_path.relative_to(GIN_CACHE_DIR)
    shrink.datalad_get_with_recovery(GIN_CACHE_DIR, rel_path)

    dest_name = shrink.normalized_dest_filename(src_path.name, MOUSE_ID, day)
    tmp_out = WORK_TMP_DIR / dest_name
    if tmp_out.exists():
        tmp_out.unlink()

    try:
        print("acquisition・processing/behavior除去中...")
        shrink.strip_to_downsampled_and_ophys(src_path, tmp_out)

        print("読み込み検証中...")
        shrink.verify_processing_only(tmp_out)

        dest_path = DEST_ROOT / dest_name
        print("Driveへコピー中...")
        shrink.copy_to_drive(tmp_out, dest_path)
        print(f"Driveへ保存: {dest_path} ({dest_path.stat().st_size / 1e6:.1f} MB)")
        summary.append((day, "converted", dest_path))
    finally:
        # 生NWB・一時processing-only版を即削除してピーク使用量を抑える
        if tmp_out.exists():
            tmp_out.unlink()
        print("生NWBをdrop中...")
        shrink.datalad_drop_with_recovery(GIN_CACHE_DIR, rel_path)

print("\n=== summary ===")
for day, status, path in summary:
    print(day, status, path)



=== task-day1 ===
既にDriveに存在するためスキップ: /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08-21_task-day1.nwb

=== task-day2 ===


GIN内でVG1GC-66 task-day2のNWBが見つかりませんでした。スキップします。

=== task-day3 ===
既にDriveに存在するためスキップ: /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08-23_task-day3.nwb

=== task-day4 ===
既にDriveに存在するためスキップ: /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08-24_task-day4.nwb

=== task-day5 ===
既にDriveに存在するためスキップ: /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08-25_task-day5.nwb

=== task-day6 ===


GIN上のパス: /home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/data/VG1GC-66/VG1GC-66_2023-08-28_task-day6.nwb
datalad get 実行中...


get(ok): data/VG1GC-66/VG1GC-66_2023-08-28_task-day6.nwb (file) [from origin...]


acquisition・processing/behavior除去中...


...

get(ok): data/VG1GC-66/VG1GC-66_2023-09-07_task-day14.nwb (file) [from origin...]
acquisition・processing/behavior除去中...


読み込み検証中...


Driveへコピー中...


Driveへ保存: /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-09-07_task-day14.nwb (184.7 MB)
生NWBをdrop中...


drop(ok): data/VG1GC-66/VG1GC-66_2023-09-07_task-day14.nwb (file) [locking origin...]

=== task-day15 ===
既にDriveに存在するためスキップ: /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-09-08_task-day15.nwb

=== summary ===
task-day1 skipped (already exists) /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08-21_task-day1.nwb
task-day2 skipped (not found on GIN) None
task-day3 skipped (already exists) /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08-23_task-day3.nwb
task-day4 skipped (already exists) /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08-24_task-day4.nwb
task-day5 skipped (already exists) /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08-25_task-day5.nwb
task-day6 converted /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08-28_task-day6.nwb
task-day7 converted /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08-29_task-day7.nwb
task-day8 converted /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08-30_task-day8.nwb
task-day9 converted /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08

## 既存day15の統一

`nwb_manual/VG1GC-66/..._task-day15.nwb`（フル版、GINから既に手動配置済みなのでGIN再取得は不要）を
processing-only版に置き換える。失敗時に元ファイルを失わないよう、一時退避してから安全に差し替える。


In [8]:
DAY15 = "task-day15"
full_path = v4.find_nwb_file(MOUSE_ID, DAY15)
print("day15の現在のファイル:", full_path)

if full_path is not None:
    shrink.check_free_space(WORK_TMP_DIR, MIN_FREE_BYTES_WSL_HOME, label="WSL側・一時ファイル")
    shrink.check_free_space(DEST_ROOT, MIN_FREE_BYTES_DRIVE, label="Drive側・置換先")

    tmp_out = WORK_TMP_DIR / full_path.name  # bdbc_nwb_explorerのセッションID解析のため元と同じ名前にする
    if tmp_out.exists():
        tmp_out.unlink()

    print("acquisition・processing/behavior除去中...")
    shrink.strip_to_downsampled_and_ophys(full_path, tmp_out)

    print("変換後ファイルの読み込み検証中...")
    shrink.verify_processing_only(tmp_out)

    print("差し替え中...")
    shrink.swap_in_place(tmp_out, full_path)

    print("差し替え後の読み込み再検証中...")
    shrink.verify_processing_only(full_path)

    shrink.finalize_swap(full_path)
    print(f"day15をprocessing-only化しました: {full_path} ({full_path.stat().st_size / 1e6:.1f} MB)")
else:
    print("day15のファイルが見つからないため、何もしません。")


day15の現在のファイル: /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-09-08_task-day15.nwb
acquisition・processing/behavior除去中...


変換後ファイルの読み込み検証中...


差し替え中...


差し替え後の読み込み再検証中...


day15をprocessing-only化しました: /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-09-08_task-day15.nwb (185.0 MB)


## 検証

- `nwb_manual/VG1GC-66/` 配下の全ファイルサイズ（フル版1.7GB程度より大幅に小さいはず）
- 各ファイルが `nwbx.read_nwb()` で読めて trials/imaging が空でないこと
- CSVがあるdayについては `v4.process_session()` が既存パイプラインとして動くこと


In [9]:
for p in sorted(DEST_ROOT.glob("*.nwb")):
    size_mb = p.stat().st_size / 1e6
    try:
        shrink.verify_processing_only(p)
        status = "OK"
    except Exception as exc:
        status = f"NG: {exc}"
    print(f"{p.name}: {size_mb:.1f} MB  [{status}]")


VG1GC-66_2023-08-21_task-day1.nwb: 185.9 MB  [OK]


VG1GC-66_2023-08-23_task-day3.nwb: 185.4 MB  [OK]


VG1GC-66_2023-08-24_task-day4.nwb: 184.9 MB  [OK]


VG1GC-66_2023-08-25_task-day5.nwb: 184.8 MB  [OK]


VG1GC-66_2023-08-28_task-day6.nwb: 185.5 MB  [OK]


...

VG1GC-66_2023-09-04_task-day11.nwb: 185.1 MB  [OK]


VG1GC-66_2023-09-05_task-day12.nwb: 185.0 MB  [OK]


VG1GC-66_2023-09-06_task-day13.nwb: 185.9 MB  [OK]


VG1GC-66_2023-09-07_task-day14.nwb: 185.0 MB  [OK]


VG1GC-66_2023-09-08_task-day15.nwb: 185.0 MB  [OK]


In [10]:
for day in dict.fromkeys(TARGET_DAYS + [DAY15]):  # TARGET_DAYSにday15が含まれる場合の重複を除く
    csv_dir = config.DATA_CSV_ROOT / MOUSE_ID / day
    if not (csv_dir / "trials_L1L2.csv").exists():
        print(f"{day}: CSVなし、スキップ")
        continue
    try:
        pack = v4.process_session(MOUSE_ID, day)
        print(f"{day}: process_session OK (n_trials={len(pack['trials'])}, has_face={pack['has_face']})")
    except Exception as exc:
        print(f"{day}: process_session NG: {exc}")


CSV読み込み中: /mnt/g/.shortcut-targets-by-id/1fI6PWRHgihU6asA4OyW-_rN-JII33Fkj/hackathon_data/VG1GC-66/task-day1/trials_L1L2.csv
全試行数: 54000
NWB読み込み中: /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08-21_task-day1.nwb


task-day1: process_session OK (n_trials=787, has_face=True)
task-day2: CSVなし、スキップ
CSV読み込み中: /mnt/g/.shortcut-targets-by-id/1fI6PWRHgihU6asA4OyW-_rN-JII33Fkj/hackathon_data/VG1GC-66/task-day3/trials_L1L2.csv
全試行数: 54000


NWB読み込み中: /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08-23_task-day3.nwb


task-day3: process_session OK (n_trials=548, has_face=True)
CSV読み込み中: /mnt/g/.shortcut-targets-by-id/1fI6PWRHgihU6asA4OyW-_rN-JII33Fkj/hackathon_data/VG1GC-66/task-day4/trials_L1L2.csv
全試行数: 54000
NWB読み込み中: /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08-24_task-day4.nwb


task-day4: process_session OK (n_trials=470, has_face=True)
CSV読み込み中: /mnt/g/.shortcut-targets-by-id/1fI6PWRHgihU6asA4OyW-_rN-JII33Fkj/hackathon_data/VG1GC-66/task-day5/trials_L1L2.csv
全試行数: 54000
NWB読み込み中: /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-08-25_task-day5.nwb


...

全試行数: 54000
NWB読み込み中: /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-09-06_task-day13.nwb


task-day13: process_session OK (n_trials=509, has_face=True)
CSV読み込み中: /mnt/g/.shortcut-targets-by-id/1fI6PWRHgihU6asA4OyW-_rN-JII33Fkj/hackathon_data/VG1GC-66/task-day14/trials_L1L2.csv


全試行数: 54000
NWB読み込み中: /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-09-07_task-day14.nwb


task-day14: process_session OK (n_trials=393, has_face=True)
CSV読み込み中: /mnt/g/.shortcut-targets-by-id/1fI6PWRHgihU6asA4OyW-_rN-JII33Fkj/hackathon_data/VG1GC-66/task-day15/trials_L1L2.csv
全試行数: 54000
NWB読み込み中: /mnt/g/マイドライブ/nwb_manual/VG1GC-66/VG1GC-66_2023-09-08_task-day15.nwb


task-day15: process_session OK (n_trials=498, has_face=True)


## 作業ディレクトリの最終掃除

WSL側の一時ファイルを消し、GINキャッシュの現在サイズを表示する。
`.git/annex/objects` が数MB程度なら生NWBは残っていない。
ext4.vhdxはこれで縮まないので、Cドライブの空きを戻したい場合は冒頭の注意点にある
`wsl --shutdown` + `Optimize-VHD` / `compact vdisk` をWindows側で実行する。

In [11]:
shrink.cleanup_work_dir(WORK_TMP_DIR)

for label, path in (("GINキャッシュ全体", GIN_CACHE_DIR), ("annex実体", GIN_CACHE_DIR / ".git" / "annex" / "objects")):
    total = sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) if path.exists() else 0
    print(f"{label}: {total / 1e6:.1f} MB ({path})")

削除対象の一時ファイルはありません: /home/takejin/gin_cache/_tmp_processing_only


GINキャッシュ全体: 5.3 MB (/home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB)
annex実体: 0.0 MB (/home/takejin/gin_cache/Kondo2025_CuedLeverPullNWB/.git/annex/objects)
